# Chunking experiments
Same BM25 search, three chunk sizes, scored by Recall@5 on the **dev** questions only. Needs no database.

In [ ]:
import sys, re
sys.path.insert(0, '..')
import numpy as np, pandas as pd
from rank_bm25 import BM25Okapi
from src.ingest import load_all

dev = pd.read_csv('../data/eval/questions.csv')
dev = dev[(dev.split == 'dev') & (dev.answerable == 'yes')]
tok = lambda t: re.findall(r'\w+', t.lower())

In [ ]:
rows = []
for size, overlap in [(100, 20), (200, 30), (400, 50)]:
    chunks = load_all('../data/docs', size, overlap)
    bm25 = BM25Okapi([tok(c['text']) for c in chunks])
    hits = 0
    for q in dev.itertuples():
        scores = bm25.get_scores(tok(q.question))
        top = [chunks[i] for i in np.argsort(scores)[::-1][:5]]
        hits += any(c['doc'] == q.doc and q.page in c['pages'] for c in top)
    rows.append({'size': size, 'overlap': overlap, 'chunks': len(chunks), 'Recall@5': round(hits / len(dev), 3)})
pd.DataFrame(rows)

## What I found
(Write 2-3 sentences: which size won, and why you think so. Then set CHUNK_SIZE in .env.)